# Model training with PyTorch Lightning

This notebook:

1. Loads prepared Iris data from parquet (same as `02_training_keras.ipynb`)
2. Splits train/test, fits `StandardScaler` on the training split, loads the `LabelEncoder` from data preparation
3. Trains a small MLP (`LightningModule`) with Adam and cross-entropy
4. Saves a Lightning checkpoint (`model.ckpt`) and `scaler.joblib` under `data/models/lightning/`


## Import libraries


In [ ]:
import os

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import lightning.pytorch as pl
from lightning.pytorch import Trainer


In [ ]:
print(f"PyTorch: {torch.__version__}")
import importlib.metadata

print(f"Lightning: {importlib.metadata.version('lightning')}")
print(f"NumPy: {np.__version__}")


## Config


In [ ]:
from config import (
    FEATURE_COLS,
    LIGHTNING_TRAINING_CHECKPOINT_PATH,
    LIGHTNING_TRAINING_DIR,
    LIGHTNING_TRAINING_INPUT_PATH,
    LIGHTNING_TRAINING_LABEL_ENCODING_PATH,
    LIGHTNING_TRAINING_SCALER_JOBLIB_PATH,
)


## Load data from parquet


In [ ]:
input_path = LIGHTNING_TRAINING_INPUT_PATH

df = pd.read_parquet(input_path)


In [ ]:
print(f"Data shape: {df.shape}")


In [ ]:
df.head()


In [ ]:
df.info()


## Prepare features and labels


In [ ]:
label_encoding_path = LIGHTNING_TRAINING_LABEL_ENCODING_PATH

feature_cols = FEATURE_COLS
label_col = "species_label"

X = df[feature_cols].values.astype(np.float32)
y = df[label_col].values.astype(np.int64)

label_encoder = joblib.load(label_encoding_path)
NB_CLASSES = len(label_encoder.classes_)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    random_state=42,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Raw features shape: {X.shape}")
print(f"Train / test (scaled): {X_train.shape}, {X_test.shape}")
print(f"Labels: train {y_train.shape}, test {y_test.shape}")
print(f"Classes: {label_encoder.classes_}")


## Define the Lightning module


In [ ]:
class IrisClassifier(pl.LightningModule):
    """Two-hidden-layer MLP matching the Keras notebook (128 → 128 → num_classes)."""

    def __init__(self, num_features: int, num_classes: int, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.net = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("train_loss", loss, prog_bar=True, on_epoch=True)
        self.log("train_acc", acc, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = nn.functional.cross_entropy(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean()
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


## DataLoaders


In [ ]:
pl.seed_everything(42, workers=True)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=16, shuffle=True
)
val_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=16)

lit_model = IrisClassifier(
    num_features=X_train.shape[1], num_classes=NB_CLASSES, lr=1e-3
)


## Train


In [ ]:
trainer = Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    log_every_n_steps=1,
    logger=False,
    enable_progress_bar=True,
)

trainer.fit(lit_model, train_loader, val_loader)


## Evaluation


In [ ]:
lit_model.eval()
with torch.no_grad():
    logits = lit_model(X_test_t)
    test_loss = nn.functional.cross_entropy(logits, y_test_t).item()
    preds = torch.argmax(logits, dim=1)
    test_acc = (preds == y_test_t).float().mean().item()

print(f"Test loss: {test_loss:.6f}")
print(f"Test accuracy: {test_acc:.6f}")


## Save checkpoint and scaler


In [ ]:
os.makedirs(LIGHTNING_TRAINING_DIR, exist_ok=True)

trainer.save_checkpoint(LIGHTNING_TRAINING_CHECKPOINT_PATH)
print(f"Checkpoint saved to: {LIGHTNING_TRAINING_CHECKPOINT_PATH}")

joblib.dump(scaler, LIGHTNING_TRAINING_SCALER_JOBLIB_PATH)
print(f"StandardScaler saved to: {LIGHTNING_TRAINING_SCALER_JOBLIB_PATH}")


## Summary


In [ ]:
print(f"✓ Checkpoint: {LIGHTNING_TRAINING_CHECKPOINT_PATH}")
print(f"✓ StandardScaler (joblib): {LIGHTNING_TRAINING_SCALER_JOBLIB_PATH}")
print("✓ LabelEncoder: use data preparation pickle (see LIGHTNING_TRAINING_LABEL_ENCODING_PATH)")
